# Project Additional Materials — Baseline AutoML (Daily)

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:** Run this notebook top to bottom in the same folder as `All_UKonly_daily_cleaned_ymd.nc`.  
This notebook trains a FLAML baseline on daily data and saves the fitted model and log.


## Step 0 — Import required libraries
Load the Python packages required for the daily FLAML baseline.  
(Dependencies are listed in the `README` and `ERP_Environment_2025.yaml`.)

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import random
import warnings
import pickle
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')

## Step 1 — Train daily baseline with FLAML
1. Set seeds.  
2. Load the preprocessed daily UK-only dataset (`All_UKonly_daily_cleaned_ymd.nc`).  
3. Define features and target as in the report.  
4. Train with k-fold CV and log to file.  
5. **Print** the best model summary and save the fitted model (pickle).

In [ ]:
# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Load dataset
ds = xr.open_dataset("All_UKonly_daily_cleaned_ymd.nc", decode_times=False)
df = ds.to_dataframe().dropna().reset_index()

# Time split
df_train = df[(df["year"] >= 2006) & (df["year"] <= 2020)]

features = ["TREFHT", "FLNS", "FSNS", "QBOT", "UBOT", "VBOT", "PRECT", "PRSN"]
target = "TREFMXAV_U"

X_train, y_train = df_train[features], df_train[target]

# FLAML baseline
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    time_budget=3600,          
    metric="r2",         
    eval_method="cv",
    n_jobs=-1,  
    log_file_name="automl_training_daily.log"
)

# Best model info
print("Best model by FLAML:", automl.best_estimator)
print("Best config:", automl.best_config)
print("Best validation loss:", automl.best_loss)

model_save_path = 'automl_training_daily.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")


[flaml.automl.logger: 08-16 14:08:43] {1752} INFO - task = regression
[flaml.automl.logger: 08-16 14:08:43] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-16 14:08:46] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 08-16 14:08:46] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost']
[flaml.automl.logger: 08-16 14:08:46] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-16 14:09:07] {2417} INFO - Estimated sufficient time budget=202949s. Estimated necessary time budget=1756s.
[flaml.automl.logger: 08-16 14:09:07] {2466} INFO -  at 52.0s,	estimator lgbm's best error=0.5294,	best estimator lgbm's best error=0.5294
[flaml.automl.logger: 08-16 14:09:07] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-16 14:09:27] {2466} INFO -  at 72.4s,	estimator lgbm's best error=0.5294,	best estimator lgbm's best error=0.5294
[flaml.automl.logger: 08-16